<a href="https://colab.research.google.com/github/Necroline9520/Proy-2026-Grupo-lX/blob/main/Copia_de_modelo_ganado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Importación de librerías y montaje de Drive

In [1]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Definir ruta de los archivos
PATH = '/content/drive/MyDrive/Colab Notebooks/archivos_modelo/'

MessageError: Error: credential propagation was unsuccessful

### 2. Consolidación de datos
Se define una función que lee los archivos y estandariza la fecha a formato `datetime` mensual. Se tienen en cuenta tanto archivos con columna explícita de fecha como archivos en formato ancho (meses por columna).

In [ ]:
archivos = [
    'cantidadterneros.xlsx', 'cantidadnovilloengorda.xlsx', 'cantidadnovillogordo.xlsx',
    'precioterneros.xlsx', 'precionovilloengorda.xlsx', 'precionovillogordo.xlsx',
    'DolarHistorico.xlsm', 'SeriesTiempoMaiz.xlsx', 'precipitacion_mensual_loslagos.xlsx'
]

def load_and_standardize(filename):
    file_path = os.path.join(PATH, filename)
    if not os.path.exists(file_path):
        print(f"Advertencia: No se encontró {filename}")
        return None

    try:
        df = pd.read_excel(file_path)
    except Exception as e:
        print(f"Error leyendo {filename}: {e}")
        return None

    meses_map = {'Ene': 1, 'Feb': 2, 'Mar': 3, 'Abr': 4, 'May': 5, 'Jun': 6,
                 'Jul': 7, 'Ago': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dic': 12,
                 'Enero': 1, 'Febrero': 2, 'Marzo': 3, 'Abril': 4, 'Mayo': 5, 'Junio': 6,
                 'Julio': 7, 'Agosto': 8, 'Septiembre': 9, 'Octubre': 10, 'Noviembre': 11, 'Diciembre': 12}

    # 1. Formato Ancho (Año y meses en columnas)
    if 'Año' in df.columns and 'Ene' in df.columns:
        cols_meses = [c for c in df.columns if c in meses_map.keys()]
        df = df.melt(id_vars=['Año'], value_vars=cols_meses, var_name='Mes_Nombre', value_name=filename.split('.')[0])
        df['Mes'] = df['Mes_Nombre'].map(meses_map)
        df['Fecha'] = pd.to_datetime(df[['Año', 'Mes']].rename(columns={'Año': 'year', 'Mes': 'month'}).assign(day=1))
        df = df.drop(columns=['Año', 'Mes', 'Mes_Nombre'])

    # 2. Formato Largo con 'Año' y 'Mes' en columnas separadas
    elif 'Año' in df.columns and 'Mes' in df.columns:
        if df['Mes'].dtype == 'O': # Si los meses están como texto
            df['Mes'] = df['Mes'].str.capitalize().str[:3].map(meses_map).fillna(df['Mes'])
        df['Fecha'] = pd.to_datetime(df[['Año', 'Mes']].rename(columns={'Año': 'year', 'Mes': 'month'}).assign(day=1))
        df = df.drop(columns=['Año', 'Mes'])

    # 3. Formato Largo con columna de fecha única
    else:
        date_cols = ['Fecha', 'Periodo', 'Año-Mes', 'date', 'fecha']
        found_col = next((col for col in df.columns if col.lower() in [c.lower() for c in date_cols]), None)

        if found_col:
            df = df.rename(columns={found_col: 'Fecha'})
            # Reemplazar abreviaciones en español si están en la columna de fecha como texto
            if df['Fecha'].dtype == 'O':
                for esp, eng in [('Ene', 'Jan'), ('Abr', 'Apr'), ('Ago', 'Aug'), ('Dic', 'Dec')]:
                    df['Fecha'] = df['Fecha'].str.replace(esp, eng, regex=False)
            df['Fecha'] = pd.to_datetime(df['Fecha']).dt.to_period('M').dt.to_timestamp()
        else:
            print(f"No se encontró columna de fecha válida en {filename}")
            return None

    return df

# Unir todos los DataFrames
df_ganado = None

for arch in archivos:
    print(f"Procesando: {arch}")
    temp_df = load_and_standardize(arch)

    if temp_df is not None:
        if df_ganado is None:
            df_ganado = temp_df
        else:
            # Merge por 'Fecha' (Outer join para conservar todos los periodos)
            df_ganado = pd.merge(df_ganado, temp_df, on='Fecha', how='outer')

if df_ganado is not None:
    df_ganado = df_ganado.sort_values('Fecha').reset_index(drop=True)
    print("\n--- Fusión (Merge) completada exitosamente ---")

### 3. Tratamiento de Precipitaciones (Media Estacional)
Se identifican los vacíos pre-2013 y se rellenan con la media mensual de 2013 en adelante.

In [ ]:
# Identificar la columna de precipitaciones
# Busca columnas que contengan 'precipitacion' u 'osorno'
columnas_precip = [c for c in df_ganado.columns if 'precip' in c.lower() or 'osorno' in c.lower()]

if columnas_precip:
    col_precip = columnas_precip[0]
    print(f"Columna de precipitaciones detectada: {col_precip}")

    # Crear columna temporal del mes para cruzar
    df_ganado['mes_temp'] = df_ganado['Fecha'].dt.month

    # Calcular promedios desde 2013
    promedios_mensuales = df_ganado[df_ganado['Fecha'].dt.year >= 2013].groupby('mes_temp')[col_precip].mean()

    # Función de relleno: si es NaN, usar el promedio del mes correspondiente
    df_ganado[col_precip] = df_ganado.apply(
        lambda row: promedios_mensuales.get(row['mes_temp']) if pd.isna(row[col_precip]) else row[col_precip],
        axis=1
    )

    # Eliminar columna temporal
    df_ganado = df_ganado.drop(columns=['mes_temp'])
    print(f"Valores nulos en '{col_precip}' rellenados con promedios estacionales.")
else:
    print("No se detectó columna de precipitaciones para aplicar tratamiento.")

### 4. Verificación y Exportación

In [ ]:
# 1. Resumen de nulos
print("=== Resumen de Valores Nulos ===")
print(df_ganado.isnull().sum())

# 2. Visualización rápida
print("\n=== Primeras 5 filas ===")
display(df_ganado.head())

# 3. Exportación a CSV
export_path = os.path.join(PATH, 'data_maestra_ganado.csv')
df_ganado.to_csv(export_path, index=False, encoding='utf-8-sig')
print(f"\nArchivo consolidado exportado exitosamente en:\n{export_path}")

### 5. Ingeniería de Variables (Feature Engineering)
Creación de variables rezagadas (lags) para capturar los ciclos biológicos del ganado y el impacto económico desfasado.

In [ ]:
# Asegurarse de que el DataFrame esté ordenado por fecha antes de aplicar shift()
df_ganado = df_ganado.sort_values('Fecha')

# 1. Rezagos Biológicos
df_ganado['cant_terneros_lag8'] = df_ganado['cantidadterneros'].shift(8)
df_ganado['cant_novillos_engorda_lag4'] = df_ganado['cantidadnovilloengorda'].shift(4)
df_ganado['cant_terneros_lag12'] = df_ganado['cantidadterneros'].shift(12)

# 2. Rezagos Económicos (Dólar Real)
col_dolar = 'Dólar ajustado a abr-2026'
df_ganado['dolar_real_lag1'] = df_ganado[col_dolar].shift(1)
df_ganado['dolar_real_lag2'] = df_ganado[col_dolar].shift(2)
df_ganado['dolar_real_lag3'] = df_ganado[col_dolar].shift(3)

# 3. Eliminar las filas con valores nulos producto de los rezagos
# Esto eliminará al menos los primeros 12 meses de historia
df_ganado_featured = df_ganado.dropna().reset_index(drop=True)

# 4. Verificar los desfases mostrando las primeras filas
print("=== DataFrame con Variables Rezagadas ===")
display(df_ganado_featured[['Fecha', 'cantidadterneros', 'cant_terneros_lag8',
                            'cant_novillos_engorda_lag4', 'cant_terneros_lag12',
                            'dolar_real_lag1']].head(10))


### 6. Corrección de Formato Numérico (Limpieza de Strings)
El dólar importado desde Excel viene con formato de texto (ej: `1.249,16`). Para que los modelos de Machine Learning funcionen, debemos reemplazar los puntos de miles, cambiar la coma decimal por un punto, y convertir la columna a tipo numérico (`float`).

In [ ]:
# Columnas que pueden tener formato de texto con comas
columnas_dolar = [
    '1. Dólar observado',
    'Dólar ajustado a abr-2026',
    'dolar_real_lag1',
    'dolar_real_lag2',
    'dolar_real_lag3'
]

for col in columnas_dolar:
    if col in df_ganado_featured.columns:
        # Verificar si la columna es de tipo objeto/texto
        if df_ganado_featured[col].dtype == 'O':
            # 1. Eliminar puntos de miles
            # 2. Reemplazar coma por punto decimal
            # 3. Convertir a tipo float numérico
            df_ganado_featured[col] = (
                df_ganado_featured[col]
                .astype(str)
                .str.replace('.', '', regex=False)
                .str.replace(',', '.', regex=False)
            )
            # Convertir a numérico, dejando como NaN lo que no se pueda convertir
            df_ganado_featured[col] = pd.to_numeric(df_ganado_featured[col], errors='coerce')

print("=== Tipos de datos corregidos ===")
print(df_ganado_featured[columnas_dolar].dtypes)

print("\n=== Primeras filas actualizadas ===")
display(df_ganado_featured[['Fecha'] + columnas_dolar].head())


### 7. Ingeniería de Variables Climáticas (Precipitaciones)
Creación de rezagos para simular el impacto en el crecimiento de pasturas y una variable binaria (flag) para eventos de lluvias extremas que afectan la logística.

In [ ]:
# Identificamos nuevamente la columna de precipitación usada previamente
col_precip = 'Osorno'

# 1. Rezagos climáticos (1 y 2 meses)
df_ganado_featured['precipitaciones_lag1'] = df_ganado_featured[col_precip].shift(1)
df_ganado_featured['precipitaciones_lag2'] = df_ganado_featured[col_precip].shift(2)

# 2. Variable binaria para lluvias extremas (Percentil 90)
# Calculamos el percentil 90 histórico de la serie de precipitaciones
percentil_90 = df_ganado_featured[col_precip].quantile(0.90)

# Asignamos 1 si supera el umbral, 0 si no
df_ganado_featured['efecto_lluvia_extrema'] = (df_ganado_featured[col_precip] > percentil_90).astype(int)

# 3. Eliminar nuevas filas con valores nulos producto de los rezagos (perderemos 2 meses más)
df_ganado_featured = df_ganado_featured.dropna().reset_index(drop=True)

# 4. Verificar resultados mostrando las últimas 5 filas
print(f"=== Umbral Lluvia Extrema (Percentil 90): {percentil_90:.1f} mm ===\n")

columnas_mostrar = ['Fecha', col_precip, 'precipitaciones_lag1',
                    'precipitaciones_lag2', 'efecto_lluvia_extrema']

print("=== Últimas 5 filas de las variables climáticas ===")
display(df_ganado_featured[columnas_mostrar].tail())


In [ ]:
# Configurar pandas para no truncar las columnas
pd.set_option('display.max_columns', None)

# Mostrar dimensiones
filas, columnas = df_ganado_featured.shape
print(f"=== Dataset Listo ===\nDimensiones: {filas} filas y {columnas} columnas\n")

# Desplegar el dataset completo
display(df_ganado_featured)


### 8. Ingeniería de Variables: Costos de Alimentación (Maíz)
Creamos 4 rezagos para modelar el costo del alimento durante los 4 meses que dura aproximadamente el ciclo de engorda.

In [ ]:
# Asumiendo que la columna se llama 'SeriesTiempoMaiz' según los pasos anteriores
col_maiz = 'SeriesTiempoMaiz'

# Aseguramos el orden por fecha
df_ganado_featured = df_ganado_featured.sort_values('Fecha')

# Crear los rezagos de 1 a 4 meses
df_ganado_featured['precio_maiz_lag1'] = df_ganado_featured[col_maiz].shift(1)
df_ganado_featured['precio_maiz_lag2'] = df_ganado_featured[col_maiz].shift(2)
df_ganado_featured['precio_maiz_lag3'] = df_ganado_featured[col_maiz].shift(3)
df_ganado_featured['precio_maiz_lag4'] = df_ganado_featured[col_maiz].shift(4)

# Eliminar los valores nulos generados por los nuevos rezagos (perderemos los primeros 4 meses)
df_ganado_featured = df_ganado_featured.dropna().reset_index(drop=True)

# Verificar las primeras 5 filas para asegurar que los rezagos están correctos
columnas_verificacion = ['Fecha', col_maiz, 'precio_maiz_lag1', 'precio_maiz_lag2', 'precio_maiz_lag3', 'precio_maiz_lag4']
print("=== Primeras 5 filas con rezagos de maíz ===")
display(df_ganado_featured[columnas_verificacion].head())


### 9. Consolidación de Ingeniería de Variables (Clima y Costos)
Bloque unificado para variables climáticas y de costos de alimentación.

In [ ]:
# Nombres de las columnas base
col_precip = 'Osorno'
col_maiz = 'SeriesTiempoMaiz'

# Asegurar el orden cronológico
df_ganado_featured = df_ganado_featured.sort_values('Fecha')

# --- 1. Variables Climáticas ---
df_ganado_featured['precipitaciones_lag1'] = df_ganado_featured[col_precip].shift(1)
df_ganado_featured['precipitaciones_lag2'] = df_ganado_featured[col_precip].shift(2)

# Alerta de lluvia extrema (Percentil 90)
percentil_90_clima = df_ganado_featured[col_precip].quantile(0.90)
df_ganado_featured['alerta_lluvia_extrema'] = (df_ganado_featured[col_precip] > percentil_90_clima).astype(int)

# --- 2. Variables de Costos (Maíz) ---
df_ganado_featured['precio_maiz_lag1'] = df_ganado_featured[col_maiz].shift(1)
df_ganado_featured['precio_maiz_lag2'] = df_ganado_featured[col_maiz].shift(2)
df_ganado_featured['precio_maiz_lag3'] = df_ganado_featured[col_maiz].shift(3)
df_ganado_featured['precio_maiz_lag4'] = df_ganado_featured[col_maiz].shift(4)

# --- 3. Limpieza de valores nulos iniciales ---
df_ganado_final = df_ganado_featured.dropna().reset_index(drop=True)

# --- 4. Verificación ---
print(f"=== Dimensiones Finales del DataFrame ===")
print(f"{df_ganado_final.shape[0]} filas y {df_ganado_final.shape[1]} columnas\n")

print("=== Primeras 5 filas ===")
display(df_ganado_final.head())

### 10. Análisis Exploratorio de Datos (EDA)
Validación de hipótesis y visualización de correlaciones.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- 1. Mapa de Calor Enfocado ---
# Definir las variables objetivo
target_vars = ['precionovilloengorda', 'precionovillogordo']

# Filtrar solo las columnas numéricas para evitar errores de correlación
numeric_cols = df_ganado_final.select_dtypes(include=[np.number]).columns.tolist()

# Excluir las variables objetivo de los predictores para visualizarlas en el eje Y
predictors = [col for col in numeric_cols if col not in target_vars]

# Calcular la matriz de correlación completa
corr_matrix = df_ganado_final[numeric_cols].corr()

# Extraer solo la intersección de Predictores vs Variables Objetivo
corr_focused = corr_matrix.loc[predictors, target_vars]

# Crear la figura
plt.figure(figsize=(10, 14))
sns.heatmap(
    corr_focused,
    annot=True,          # Anotar valores numéricos
    cmap='coolwarm',     # Paleta divergente (azul a rojo)
    vmin=-1, vmax=1,     # Rango de la correlación
    fmt=".2f",           # Formato a 2 decimales
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)

plt.title('Correlación de Variables Predictoras vs Precios del Ganado', fontsize=16, pad=20)
plt.xlabel('Variables Objetivo', fontsize=12)
plt.ylabel('Variables Predictoras (Lags, Clima, Costos, etc.)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2. Gráfico de Dispersión con Línea de Tendencia ---
plt.figure(figsize=(10, 6))

# regplot dibuja el scatterplot y ajusta un modelo de regresión lineal (línea de tendencia)
sns.regplot(
    data=df_ganado_final,
    x='cant_terneros_lag12',
    y='precionovillogordo',
    scatter_kws={'alpha': 0.6, 'color': '#2b7bba'}, # Puntos semi-transparentes
    line_kws={'color': 'red', 'linewidth': 2}       # Línea de tendencia en rojo
)

plt.title('Efecto Oferta-Precio: Terneros (hace 12 meses) vs Precio Novillo Gordo (Actual)', fontsize=14, pad=15)
plt.xlabel('Cantidad de Terneros (Rezago 12 meses)', fontsize=12)
plt.ylabel('Precio Novillo Gordo ($)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 11. Prevención de Data Leakage y Estacionariedad
Eliminamos variables contemporáneas que no conoceríamos en la vida real antes de predecir, y calculamos las variaciones porcentuales para hacer las series estacionarias y evitar la ilusión inflacionaria.

In [ ]:
# 1. Crear una copia del DataFrame final para modelado
df_ganado_ml = df_ganado_final.copy()

# 2. Selección y Descarte de Variables (Data Leakage y Ruido)
# Variables a eliminar
vars_to_drop = [
    'precioterneros', 'cantidadterneros', 'cantidadnovilloengorda', 'cantidadnovillogordo', # Contemporáneas
    '1. Dólar observado',                                                                    # Nominal/Inflación
    'precio_maiz_lag1', 'precio_maiz_lag2', 'precio_maiz_lag3', 'precio_maiz_lag4'           # Rezagos de maíz (ruido)
]

# Eliminar si existen en el DataFrame
df_ganado_ml = df_ganado_ml.drop(columns=[col for col in vars_to_drop if col in df_ganado_ml.columns])

# 3. Transformación para Estacionariedad (Variación Porcentual)
# Ordenar por fecha por seguridad
df_ganado_ml = df_ganado_ml.sort_values('Fecha')

# Variación porcentual mensual (Retornos)
df_ganado_ml['var_precionovillogordo'] = df_ganado_ml['precionovillogordo'].pct_change()
df_ganado_ml['var_precionovilloengorda'] = df_ganado_ml['precionovilloengorda'].pct_change()
df_ganado_ml['var_dolar_real'] = df_ganado_ml['Dólar ajustado a abr-2026'].pct_change()

# 4. Limpieza Final
# Eliminar la primera fila que quedará con nulos tras el pct_change()
df_ganado_ml = df_ganado_ml.dropna().reset_index(drop=True)

# 5. Verificación
print("=== Columnas Finales para el Modelo ===")
for col in df_ganado_ml.columns:
    print(f"- {col}")

print("\n=== Primeras 5 filas del Dataset de Machine Learning ===")
display(df_ganado_ml.head())

### 12. Train/Test Split Cronológico
Separación de datos en conjuntos de Entrenamiento y Prueba respetando el orden temporal de la serie (sin selección aleatoria).

In [ ]:
# 1. Definir Variable Objetivo y Características (Features)
target = 'var_precionovillogordo'

# Seleccionamos estrictamente los rezagos (lags) ya que representan el pasado conocido.
# Excluimos cualquier variable contemporánea para evitar el data leakage.
X_cols = [col for col in df_ganado_ml.columns if 'lag' in col.lower()]

X = df_ganado_ml[X_cols]
y = df_ganado_ml[target]
fechas = df_ganado_ml['Fecha']  # Guardamos la fecha para la validación de periodos

# 2. Split Cronológico (80% Entrenamiento, 20% Prueba)
split_index = int(len(df_ganado_ml) * 0.8)

# Datos de Entrenamiento (hasta el split_index)
X_train = X.iloc[:split_index]
y_train = y.iloc[:split_index]
fechas_train = fechas.iloc[:split_index]

# Datos de Prueba (desde el split_index hasta el final)
X_test = X.iloc[split_index:]
y_test = y.iloc[split_index:]
fechas_test = fechas.iloc[split_index:]

# 3. Verificación de Dimensiones y Fechas
print("=== Variables Predictoras Incluidas (X) ===")
print(X_cols)

print("\n=== Dimensiones de los Conjuntos ===")
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}  | y_test:  {y_test.shape}")

print("\n=== Periodos de Tiempo (Sin Solapamiento) ===")
print(f"Entrenamiento (Train): Desde {fechas_train.min().strftime('%Y-%m-%d')} hasta {fechas_train.max().strftime('%Y-%m-%d')}")
print(f"Prueba (Test):         Desde {fechas_test.min().strftime('%Y-%m-%d')} hasta {fechas_test.max().strftime('%Y-%m-%d')}")

### 13. Entrenamiento y Evaluación del Modelo (XGBoost)
Entrenamiento del regresor con hiperparámetros conservadores para evitar el sobreajuste, evaluación de métricas de error y cálculo de la capacidad del modelo para predecir la dirección del mercado (Accuracy Direccional).

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# 1. Instanciar y Entrenar el Modelo XGBoost
# Usamos hiperparámetros conservadores sugeridos para evitar overfitting en series de tiempo cortas
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

xgb_model.fit(X_train, y_train)

# 2. Generar Predicciones sobre el conjunto de prueba (Test)
y_pred = xgb_model.predict(X_test)

# 3. Evaluar Métricas
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# Accuracy Direccional: % de veces que el modelo acertó la dirección (signo de la variación)
# np.sign() devuelve -1 si es negativo, 0 si es 0, y 1 si es positivo
accuracy_direccional = np.mean(np.sign(y_test) == np.sign(y_pred)) * 100

print("=== Evaluación del Modelo XGBoost (Conjunto de Prueba) ===")
print(f"MAE (Error Absoluto Medio): {mae:.4f}")
print(f"RMSE (Error Cuadrático Medio): {rmse:.4f}")
print(f"Accuracy Direccional:       {accuracy_direccional:.1f}%")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 4. Visualización: Variación Real vs Predicha
plt.figure(figsize=(14, 6))

# Usamos las fechas del test para el eje X
plt.plot(fechas_test, y_test, label='Variación Real', color='#1f77b4', linewidth=2, marker='o', markersize=4)
plt.plot(fechas_test, y_pred, label='Variación Predicha (XGBoost)', color='#ff7f0e', linestyle='--', linewidth=2, marker='X', markersize=5)

# Línea horizontal punteada en 0 para identificar fácilmente subidas vs bajadas
plt.axhline(0, color='red', linestyle=':', linewidth=2, alpha=0.7, label='Línea Cero (Punto de Inflexión)')

plt.title('Variación Porcentual del Precio del Novillo Gordo: Real vs Predicción', fontsize=15, pad=15)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Variación Porcentual Mensual', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='best', fontsize=11)
plt.tight_layout()

plt.show()

### 14. Reentrenamiento con Regularización Fuerte y Clipping
Aplicamos hiperparámetros estrictos para evitar el sobreajuste y limitamos las predicciones a los máximos y mínimos históricos conocidos para evitar variaciones irreales.

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Entrenamiento con Regularización Moderada (Corrigiendo el Underfitting)
xgb_model_reg = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,           # Profundidad moderada
    reg_lambda=1,          # Penalización L2 moderada
    reg_alpha=0.1,         # Penalización L1 suave
    min_child_weight=1,    # Peso normal por hoja
    random_state=42
)

xgb_model_reg.fit(X_train, y_train)

# 2. Generar predicciones iniciales
y_pred_raw = xgb_model_reg.predict(X_test)

# 3. Clipping (Acotamiento Realista)
limite_inferior = y_train.min()
limite_superior = y_train.max()

y_pred_clipped = np.clip(y_pred_raw, limite_inferior, limite_superior)

# 4. Nuevas Métricas
mae_clipped = mean_absolute_error(y_test, y_pred_clipped)
rmse_clipped = np.sqrt(mean_squared_error(y_test, y_pred_clipped))
accuracy_direccional_clipped = np.mean(np.sign(y_test) == np.sign(y_pred_clipped)) * 100

print("=== Evaluación del Modelo XGBoost (Regularizado Moderado y Acotado) ===")
print(f"Limites Históricos aplicados: Min: {limite_inferior:.4f} | Max: {limite_superior:.4f}")
print(f"MAE: {mae_clipped:.4f}")
print(f"RMSE: {rmse_clipped:.4f}")
print(f"Accuracy Direccional: {accuracy_direccional_clipped:.1f}%")

# 5. Nuevo Gráfico de Predicción vs Realidad
plt.figure(figsize=(14, 6))
plt.plot(fechas_test, y_test, label='Variación Real', color='#1f77b4', linewidth=2, marker='o', markersize=4)
plt.plot(fechas_test, y_pred_clipped, label='Variación Predicha (Moderada + Clipped)', color='#2ca02c', linestyle='--', linewidth=2, marker='X', markersize=5)
plt.axhline(0, color='red', linestyle=':', linewidth=2, alpha=0.7, label='Línea Cero')

plt.title('Variación Porcentual: Real vs Predicción Regularizada y Acotada', fontsize=15, pad=15)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Variación Porcentual Mensual', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 6. Auditoría de Importancia de Variables (Feature Importance)
importancias = xgb_model_reg.feature_importances_

# Crear un DataFrame para facilitar la visualización
df_importancias = pd.DataFrame({
    'Variable': X_cols,
    'Importancia': importancias
}).sort_values(by='Importancia', ascending=False)

# Imprimir el Top 3
print("\n=== Top 3 Variables Más Importantes ===")
for i, (var, imp) in enumerate(zip(df_importancias['Variable'].head(3), df_importancias['Importancia'].head(3))):
    print(f"{i+1}. {var} ({imp:.4f})")

# Gráfico de barras horizontal para el Top 10 (o todas si son menos)
plt.figure(figsize=(10, 6))
sns.barplot(
    x='Importancia',
    y='Variable',
    data=df_importancias.head(10),
    palette='viridis',
    hue='Variable',
    legend=False
)

plt.title('Importancia de Variables (Feature Importance) - XGBoost Moderado', fontsize=14, pad=15)
plt.xlabel('Importancia Relativa', fontsize=12)
plt.ylabel('Variable Predictora', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### 15. Entrenamiento y Evaluación con Random Forest
Entrenamos un modelo alternativo basado en ensambles (Random Forest) para comparar su rendimiento predictivo y su selección de variables frente al XGBoost.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd # Added import for pandas

# 1. Instanciar y Entrenar el Modelo Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=4,           # Ajustado a profundidad 3 para mejor Accuracy Direccional, siguiendo la indicación del comentario
    min_samples_leaf=3,    # Aumentamos el número de muestras mínimas por hoja
    random_state=42
)

rf_model.fit(X_train, y_train)

# 2. Generar Predicciones sobre el conjunto de prueba (Test)
y_pred_rf = rf_model.predict(X_test)

# --- INICIO DE LÓGICA PARA RESTRICCIONES DE PREDICCIONES (MES DE ENERO) ---

# Obtener los límites históricos generales del conjunto de entrenamiento
limite_inferior_general = y_train.min()
limite_superior_general = y_train.max()

# Calcular el límite superior específico para el mes de Enero en los datos de entrenamiento
# Esto ayudará a controlar las predicciones "descontroladas" de Enero
# Asegurarse de que 'fechas_train' esté disponible y sea un objeto Series con un dt accessor.
january_y_train = y_train[fechas_train.dt.month == 1]
# Si no hay datos de enero en el entrenamiento, se usa el límite superior general.
# Modificación: Usamos el percentil 75 en lugar del máximo para un clipping más restrictivo en enero.
limite_superior_enero = january_y_train.quantile(0.75) if not january_y_train.empty else limite_superior_general

# Crear una copia de las predicciones para aplicar el clipping
y_pred_rf_clipped = y_pred_rf.copy()

# Aplicar clipping condicional:
# Si la predicción es para Enero, usar el limite_superior_enero (más estricto si january_y_train.max() es menor que el general)
# Si no es Enero, usar el limite_superior_general
for i, fecha in enumerate(fechas_test):
    if fecha.month == 1: # Si la fecha de la predicción es Enero
        y_pred_rf_clipped[i] = np.clip(y_pred_rf_clipped[i], limite_inferior_general, limite_superior_enero)
    else: # Para cualquier otro mes
        y_pred_rf_clipped[i] = np.clip(y_pred_rf_clipped[i], limite_inferior_general, limite_superior_general)

# --- FIN DE LÓGICA PARA RESTRICCIONES DE PREDICCIONES (MES DE ENERO) ---


# 3. Evaluar Métricas
mae_rf = mean_absolute_error(y_test, y_pred_rf_clipped)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf_clipped))
accuracy_direccional_rf = np.mean(np.sign(y_test) == np.sign(y_pred_rf_clipped)) * 100

print("=== Evaluación del Modelo Random Forest ===")
print(f"MAE: {mae_rf:.4f}")
print(f"RMSE: {rmse_rf:.4f}")
print(f"Accuracy Direccional: {accuracy_direccional_rf:.1f}%")

# 4. Gráfico de Predicción vs Realidad
plt.figure(figsize=(14, 6))
plt.plot(fechas_test, y_test, label='Variación Real', color='#1f77b4', linewidth=2, marker='o', markersize=4)
plt.plot(fechas_test, y_pred_rf_clipped, label='Variación Predicha (Random Forest)', color='#9467bd', linestyle='--', linewidth=2, marker='s', markersize=5)
plt.axhline(0, color='red', linestyle=':', linewidth=2, alpha=0.7, label='Línea Cero')

plt.title('Variación Porcentual: Real vs Predicción (Random Forest)', fontsize=15, pad=15)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Variación Porcentual Mensual', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns

# 5. Auditoría de Importancia de Variables (Random Forest)
importancias_rf = rf_model.feature_importances_

# Crear un DataFrame para facilitar la visualización
df_importancias_rf = pd.DataFrame({
    'Variable': X_cols,
    'Importancia': importancias_rf
}).sort_values(by='Importancia', ascending=False)

# Imprimir el Top 3
print("\n=== Top 3 Variables Más Importantes (Random Forest) ===")
for i, (var, imp) in enumerate(zip(df_importancias_rf['Variable'].head(3), df_importancias_rf['Importancia'].head(3))):
    print(f"{i+1}. {var} ({imp:.4f})")

# Gráfico de barras horizontal
plt.figure(figsize=(10, 6))
sns.barplot(
    x='Importancia',
    y='Variable',
    data=df_importancias_rf.head(10),
    palette='magma',
    hue='Variable',
    legend=False
)

plt.title('Importancia de Variables (Feature Importance) - Random Forest', fontsize=14, pad=15)
plt.xlabel('Importancia Relativa', fontsize=12)
plt.ylabel('Variable Predictora', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### 16. Optimización del Modelo Random Forest (Hyperparameter Tuning con GridSearchCV)

Para 'continuar entrenando' un modelo de Random Forest y mejorar su rendimiento, buscamos los mejores hiperparámetros. `GridSearchCV` nos permite probar sistemáticamente diferentes combinaciones de hiperparámetros y encontrar la que ofrece el mejor rendimiento según una métrica de evaluación definida.

Algunos hiperparámetros clave para optimizar en Random Forest son:
-   `n_estimators`: El número de árboles en el bosque.
-   `max_depth`: La profundidad máxima de cada árbol.
-   `min_samples_leaf`: El número mínimo de muestras requeridas para estar en un nodo hoja.
-   `max_features`: El número de características a considerar al buscar la mejor división.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

print("--- Iniciando Búsqueda de Hiperparámetros con GridSearchCV para Random Forest ---")

# 1. Definir la rejilla de parámetros a buscar
# Estos rangos pueden ajustarse según el conocimiento del dominio y la capacidad computacional.
param_grid = {
    'n_estimators': [50, 100, 150],  # Número de árboles
    'max_depth': [None, 5, 8],       # Profundidad máxima del árbol
    'min_samples_leaf': [1, 2, 4],   # Mínimo de muestras en un nodo hoja
    'max_features': [0.7, 0.9, 1.0] # Proporción de características a considerar para cada división
}

# 2. Instanciar el modelo base de Random Forest
rf = RandomForestRegressor(random_state=42)

# 3. Configurar GridSearchCV
# cv=3 indica validación cruzada de 3 splits.
# scoring='neg_mean_absolute_error' para buscar el menor MAE (GridSearchCV minimiza, por eso 'neg_').
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,  # Usar todos los núcleos disponibles
    scoring='neg_mean_absolute_error', # Para optimizar MAE
    verbose=1
)

# 4. Ejecutar la búsqueda en la rejilla
grid_search.fit(X_train, y_train)

# 5. Obtener los mejores parámetros y el mejor score
best_params = grid_search.best_params_
best_mae_score = -grid_search.best_score_ # Convertir a MAE positivo

print(f"\n--- Mejores Hiperparámetros Encontrados: {best_params} ---")
print(f"--- Mejor MAE (Validación Cruzada): {best_mae_score:.4f} ---")

# 6. Obtener el mejor modelo entrenado
rf_model_tuned = grid_search.best_estimator_

# 7. Generar Predicciones sobre el conjunto de prueba (Test) con el modelo optimizado
y_pred_rf_tuned = rf_model_tuned.predict(X_test)

# 8. Clipping (Acotamiento Realista) - Misma lógica que antes
limite_inferior_general = y_train.min()
limite_superior_general = y_train.max()

january_y_train = y_train[fechas_train.dt.month == 1]
limite_superior_enero = january_y_train.quantile(0.75) if not january_y_train.empty else limite_superior_general

y_pred_rf_tuned_clipped = y_pred_rf_tuned.copy()
for i, fecha in enumerate(fechas_test):
    if fecha.month == 1:
        y_pred_rf_tuned_clipped[i] = np.clip(y_pred_rf_tuned_clipped[i], limite_inferior_general, limite_superior_enero)
    else:
        y_pred_rf_tuned_clipped[i] = np.clip(y_pred_rf_tuned_clipped[i], limite_inferior_general, limite_superior_general)

# 9. Evaluar Métricas del modelo optimizado
mae_rf_tuned = mean_absolute_error(y_test, y_pred_rf_tuned_clipped)
rmse_rf_tuned = np.sqrt(mean_squared_error(y_test, y_pred_rf_tuned_clipped))
accuracy_direccional_rf_tuned = np.mean(np.sign(y_test) == np.sign(y_pred_rf_tuned_clipped)) * 100
r2_rf_tuned = r2_score(y_test, y_pred_rf_tuned_clipped)

print("\n=== Evaluación del Modelo Random Forest Optimizado ===")
print(f"MAE: {mae_rf_tuned:.4f}")
print(f"RMSE: {rmse_rf_tuned:.4f}")
print(f"R2 Score: {r2_rf_tuned:.4f}")
print(f"Accuracy Direccional: {accuracy_direccional_rf_tuned:.1f}%")

# 10. Gráfico de Predicción vs Realidad para el modelo optimizado
plt.figure(figsize=(14, 6))
plt.plot(fechas_test, y_test, label='Variación Real', color='#1f77b4', linewidth=2, marker='o', markersize=4)
plt.plot(fechas_test, y_pred_rf_tuned_clipped, label='Variación Predicha (RF Optimizado)', color='#e377c2', linestyle='--', linewidth=2, marker='^', markersize=5)
plt.axhline(0, color='red', linestyle=':', linewidth=2, alpha=0.7, label='Línea Cero')

plt.title('Variación Porcentual: Real vs Predicción (Random Forest Optimizado)', fontsize=15, pad=15)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Variación Porcentual Mensual', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 11. Auditoría de Importancia de Variables (Random Forest Optimizado)
importancias_rf_tuned = rf_model_tuned.feature_importances_

# Crear un DataFrame para facilitar la visualización
df_importancias_rf_tuned = pd.DataFrame({
    'Variable': X_cols,
    'Importancia': importancias_rf_tuned
}).sort_values(by='Importancia', ascending=False)

# Imprimir el Top 3
print("\n=== Top 3 Variables Más Importantes (Random Forest Optimizado) ===")
for i, (var, imp) in enumerate(zip(df_importancias_rf_tuned['Variable'].head(3), df_importancias_rf_tuned['Importancia'].head(3))):
    print(f"{i+1}. {var} ({imp:.4f})")

# Gráfico de barras horizontal
plt.figure(figsize=(10, 6))
sns.barplot(
    x='Importancia',
    y='Variable',
    data=df_importancias_rf_tuned.head(10),
    palette='viridis',
    hue='Variable',
    legend=False
)

plt.title('Importancia de Variables (Feature Importance) - Random Forest Optimizado', fontsize=14, pad=15)
plt.xlabel('Importancia Relativa', fontsize=12)
plt.ylabel('Variable Predictora', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import r2_score

# --- Evaluación R2 para XGBoost ---
r2_xgb = r2_score(y_test, y_pred_clipped)

# --- Evaluación R2 para Random Forest ---
r2_rf = r2_score(y_test, y_pred_rf_clipped)

print("=== Resumen Comparativo de Métricas del Modelo ===")
print("---------------------------------------------------")
print("| Modelo           | MAE      | RMSE     | R2       | Accuracy Direccional |")
print("---------------------------------------------------")
print(f"| XGBoost          | {mae_clipped:.4f} | {rmse_clipped:.4f} | {r2_xgb:.4f} | {accuracy_direccional_clipped:.1f}%              |")
print(f"| Random Forest    | {mae_rf:.4f} | {rmse_rf:.4f} | {r2_rf:.4f} | {accuracy_direccional_rf:.1f}%              |")
print("---------------------------------------------------")

### 16. Evaluación Rápida de Múltiples Modelos con `LazyPredict`

Para comparar rápidamente el rendimiento de muchos algoritmos de Machine Learning en tu conjunto de datos, podemos utilizar la librería `LazyPredict`. Esta herramienta automatiza el proceso de entrenar y evaluar múltiples modelos, proporcionando una tabla comparativa de sus métricas clave.

In [ ]:
# Instalar la librería LazyPredict si no está instalada
!pip install lazypredict

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from lazypredict.Supervised import LazyRegressor
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Carga y Preparación de Datos (Ejemplo Simulado) ---
# Dado que la solicitud es para un ejemplo, simularemos un DataFrame.
# En un escenario real, aquí cargarías tu df_ganado_ml.
print("--- Generando Datos de Ejemplo Simulados ---")

# Definir un rango de fechas para el ejemplo
fechas_ejemplo = pd.date_range(start='2000-01-01', periods=200, freq='M')

# Crear datos aleatorios para las características
np.random.seed(42) # Para reproducibilidad
data = {
    'precio_historico': np.random.rand(200) * 1000 + 500, # Precio base
    'stock_ganado': np.random.randint(500, 2000, 200),     # Cantidad de ganado
    'tipo_de_cambio': np.random.rand(200) * 800 + 700,    # Tipo de cambio (ej. USD a CLP)
    'importaciones': np.random.randint(50, 300, 200),      # Volumen de importaciones
    'clima': np.random.rand(200) * 30 + 10,               # Índice climático (ej. temperatura promedio)
    'precipitaciones_lag1': np.random.rand(200) * 200,    # Precipitaciones del mes anterior
    'dolar_real_lag1': np.random.rand(200) * 800 + 700    # Dólar real del mes anterior
}
df_simulado = pd.DataFrame(data, index=fechas_ejemplo)

# Generar la variable objetivo 'precio_futuro'
# Una relación simple para el ejemplo
df_simulado['precio_futuro'] = (
    0.5 * df_simulado['precio_historico'] +
    -0.1 * df_simulado['stock_ganado'] +
    0.2 * df_simulado['tipo_de_cambio'] +
    -0.05 * df_simulado['importaciones'] +
    0.3 * df_simulado['clima'] +
    np.random.randn(200) * 50 # Añadir algo de ruido
)

# En tu caso, usarías df_ganado_ml para X y la variable objetivo 'var_precionovillogordo' para y
# X = df_ganado_ml.drop(columns=['Fecha', 'var_precionovillogordo', 'precionovilloengorda', 'precionovillogordo', 'Dólar ajustado a abr-2026', 'SeriesTiempoMaiz', 'Osorno'])
# y = df_ganado_ml['var_precionovillogordo']

# Para este ejemplo:
X_simulado = df_simulado.drop(columns=['precio_futuro'])
y_simulado = df_simulado['precio_futuro']

print("Dataset simulado creado:")
display(df_simulado.head())

# --- 2. División de Datos (Cronológica para series de tiempo) ---
# Dividimos los datos en conjuntos de entrenamiento y prueba respetando el orden temporal.

split_index_simulado = int(len(df_simulado) * 0.8)

X_train = X_simulado.iloc[:split_index_simulado]
y_train = y_simulado.iloc[:split_index_simulado]

X_test = X_simulado.iloc[split_index_simulado:]
y_test = y_simulado.iloc[split_index_simulado:]

# Guardamos las fechas para los gráficos, que serán el índice de y_test
fechas_test = y_test.index

print(f"\nDimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test: {X_test.shape}")
print(f"Dimensiones de y_train: {y_train.shape}")
print(f"Dimensiones de y_test: {y_test.shape}")

In [ ]:
# --- 3. Ejecución de LazyRegressor ---
# Inicializamos LazyRegressor y le pedimos que entrene y evalúe los modelos.
# verbose=0 para no imprimir cada modelo en consola.
# ignore_warnings=True para suprimir advertencias de algunos modelos.
print("\n--- Ejecutando LazyRegressor (esto puede tardar unos minutos) ---")

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)

# Entrenar y predecir con todos los modelos disponibles
models, predictions = reg.fit(X_train, X_test, y_train, y_test)

print("\n--- Tabla Comparativa de Modelos ---")
# La variable 'models' es un DataFrame que contiene las métricas de todos los modelos.
# Lo ordenamos por R-squared de mayor a menor para ver los mejores rendimientos.
display(models.sort_values(by='R-Squared', ascending=False))


In [ ]:
# --- 4. Visualización Básica: R-squared de los Top 10 Modelos ---
print("\n--- Visualizando el rendimiento de los Top 10 modelos (R-squared) ---")

# Seleccionamos los 10 mejores modelos según R-squared
top_10_models = models.sort_values(by='R-Squared', ascending=False).head(10)

plt.figure(figsize=(12, 8))
sns.barplot(
    x='R-Squared',
    y=top_10_models.index, # Los nombres de los modelos están en el índice
    data=top_10_models,
    palette='viridis', # Una paleta de colores atractiva
    hue=top_10_models.index,
    legend=False
)
plt.title('Top 10 Modelos por R-squared (LazyPredict)', fontsize=16, pad=15)
plt.xlabel('R-Squared', fontsize=12)
plt.ylabel('Modelo', fontsize=12)
plt.xlim(0, 1) # R-squared va de 0 a 1 (o puede ser negativo)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("\n¡LazyPredict ha completado la evaluación rápida de los modelos!")

### 17. Ejecución de LazyPredict con Datos Reales

Ahora vamos a utilizar `LazyRegressor` con los conjuntos de entrenamiento y prueba (`X_train`, `X_test`, `y_train`, `y_test`) que hemos preparado previamente en el cuaderno.

In [ ]:
from lazypredict.Supervised import LazyRegressor
import pandas as pd

print("\n--- Ejecutando LazyRegressor con datos reales (esto puede tardar unos minutos) ---")

# Inicializamos LazyRegressor
reg_real_data = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)

# Entrenar y predecir con todos los modelos disponibles usando los datos reales
models_real, predictions_real = reg_real_data.fit(X_train, X_test, y_train, y_test)

print("\n--- Tabla Comparativa de Modelos (Datos Reales) ---")
# Mostramos la tabla resumen de resultados ordenada por R-squared
display(models_real.sort_values(by='R-Squared', ascending=False))

### 18. Entrenamiento y Evaluación de Ridge Regression

Vamos a aplicar un modelo de Ridge Regression, un tipo de regresión lineal regularizada, a nuestros datos para comparar su rendimiento con los modelos anteriores. Ridge Regression ayuda a prevenir el sobreajuste al penalizar la magnitud de los coeficientes.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd # Asegurarse de que pandas esté importado

print("\n--- Entrenando y evaluando Ridge Regression ---")

# RE-DEFINIR CONJUNTOS DE ENTRENAMIENTO Y PRUEBA CON DATOS REALES
# Esto asegura que no se usen los datos simulados de LazyPredict.
# Las variables 'target', 'X_cols' y 'df_ganado_ml' deberían estar disponibles
# desde pasos anteriores (celda 12. Train/Test Split Cronológico).

# Si X_cols o target no estuvieran definidos, se deberían regenerar:
# target = 'var_precionovillogordo'
# X_cols = [col for col in df_ganado_ml.columns if 'lag' in col.lower()]

X = df_ganado_ml[X_cols]
y = df_ganado_ml[target]
fechas = df_ganado_ml['Fecha']

split_index = int(len(df_ganado_ml) * 0.8)

X_train = X.iloc[:split_index]
y_train = y.iloc[:split_index]
fechas_train = fechas.iloc[:split_index]

X_test = X.iloc[split_index:]
y_test = y.iloc[split_index:]
fechas_test = fechas.iloc[split_index:]
# FIN DE RE-DEFINICIÓN DE DATOS REALES

# 1. Importar e instanciar el modelo Ridge Regression
ridge_model = Ridge(random_state=42)

# 2. Ajustar (entrenar) el modelo con los datos de entrenamiento
ridge_model.fit(X_train, y_train)

# 3. Realizar las predicciones sobre el conjunto de prueba
y_pred_ridge = ridge_model.predict(X_test)

# 4. Calcular e imprimir las métricas finales
r2_ridge = r2_score(y_test, y_pred_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
accuracy_direccional_ridge = np.mean(np.sign(y_test) == np.sign(y_pred_ridge)) * 100 # Moved from cell 10b780fe

print("\n=== Métricas de Evaluación para Ridge Regression ===")
print(f"R² (Coeficiente de Determinación): {r2_ridge:.4f}")
print(f"MAE (Error Absoluto Medio):         {mae_ridge:.4f}")
print(f"RMSE (Raíz del Error Cuadrático Medio): {rmse_ridge:.4f}")
print(f"Accuracy Direccional:       {accuracy_direccional_ridge:.1f}%")


### 19. Análisis de Residuos para Ridge Regression

El análisis de residuos es crucial para validar las suposiciones del modelo de regresión. Un buen modelo debería tener residuos distribuidos aleatoriamente alrededor de cero, sin patrones evidentes. Visualizaremos los residuos para el modelo Ridge.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calcular los residuos
# Los residuos son la diferencia entre los valores reales (y_test) y los valores predichos (y_pred_ridge)
residuos_ridge = y_test - y_pred_ridge

# 2. Generar el gráfico de dispersión de Predicciones vs Residuos
plt.figure(figsize=(12, 7))
sns.scatterplot(x=y_pred_ridge, y=residuos_ridge, alpha=0.7, color='#2ca02c')

# 3. Dibujar una línea horizontal de referencia en y=0
plt.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Residuo = 0')

# 4. Incluir títulos, etiquetas y cuadrícula
plt.title('Análisis de Residuos del Modelo Ridge Regression', fontsize=16, pad=15)
plt.xlabel('Valores Predichos por Ridge Regression', fontsize=12)
plt.ylabel('Residuos (Real - Predicho)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

### 20. Visualización: Variación Real vs Predicción (Ridge Regression)

Este gráfico de serie temporal nos permite comparar visualmente cómo el modelo de Ridge Regression sigue la 'Variación Real' del precio del novillo gordo a lo largo del tiempo en el conjunto de prueba.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 7))

# Variación Real (y_test)
plt.plot(fechas_test, y_test, label='Variación Real', color='#1f77b4', linewidth=2, marker='o', markersize=4)

# Variación Predicha (y_pred_ridge)
plt.plot(fechas_test, y_pred_ridge, label='Variación Predicha (Ridge Regression)', color='#9467bd', linestyle='--', linewidth=2, marker='s', markersize=5)

# Línea Cero de referencia
plt.axhline(0, color='red', linestyle=':', linewidth=2, alpha=0.7, label='Línea Cero')

plt.title('Variación Porcentual: Real vs Predicción (Ridge Regression)', fontsize=15, pad=15)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Variación Porcentual Mensual', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

### 21. Importancia de Variables (Feature Importance para Ridge Regression)

Para el modelo Ridge, la importancia de las variables se deriva de la magnitud absoluta de sus coeficientes. Un coeficiente más grande (en valor absoluto) indica una mayor influencia de la variable en la predicción. Visualicemos las variables más influyentes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge # Re-importar Ridge para asegurar consistencia

# Re-instanciar y re-entrenar el modelo Ridge para asegurar que los coeficientes coincidan con X_train
# Esto es una medida de precaución para evitar desincronizaciones del estado del kernel
ridge_model = Ridge(random_state=42)
ridge_model.fit(X_train, y_train)

# Coeficientes del modelo Ridge
# La importancia se mide por el valor absoluto de los coeficientes
importancias_ridge = np.abs(ridge_model.coef_)

# Crear un DataFrame para facilitar la visualización
# X_cols contiene los nombres de las columnas de las variables predictoras
df_importancias_ridge = pd.DataFrame({
    'Variable': X_cols,
    'Importancia': importancias_ridge
}).sort_values(by='Importancia', ascending=False)

# Imprimir el Top 3
print("\n=== Top 3 Variables Más Importantes (Ridge Regression) ===")
for i, (var, imp) in enumerate(zip(df_importancias_ridge['Variable'].head(3), df_importancias_ridge['Importancia'].head(3))):
    print(f"{i+1}. {var} ({imp:.4f})")

# Gráfico de barras horizontal para el Top 10 (o todas si son menos)
plt.figure(figsize=(10, 6))
sns.barplot(
    x='Importancia',
    y='Variable',
    data=df_importancias_ridge.head(10),
    palette='magma',
    hue='Variable',
    legend=False
)

plt.title('Importancia de Variables (Feature Importance) - Ridge Regression', fontsize=14, pad=15)
plt.xlabel('Importancia Relativa (Valor Absoluto del Coeficiente)', fontsize=12)
plt.ylabel('Variable Predictora', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### 22. Resumen Comparativo de Métricas de Modelos

Aquí se presenta una tabla consolidada con las métricas clave de rendimiento para los modelos XGBoost, Random Forest y Ridge Regression que hemos entrenado. Esto nos permite comparar su efectividad de un vistazo.

In [ ]:
# Ridge Regression metrics are already calculated as r2_ridge, mae_ridge, rmse_ridge

print("=== Resumen Comparativo de Métricas del Modelo ===")
print("---------------------------------------------------")
print("| Modelo           | MAE      | RMSE     | R2       | Accuracy Direccional |")
print("---------------------------------------------------")
print(f"| XGBoost          | {mae_clipped:.4f} | {rmse_clipped:.4f} | {r2_xgb:.4f} | {accuracy_direccional_clipped:.1f}%              |")
print(f"| Random Forest    | {mae_rf:.4f} | {rmse_rf:.4f} | {r2_rf:.4f} | {accuracy_direccional_rf:.1f}%              |")
print(f"| Ridge Regression | {mae_ridge:.4f} | {rmse_ridge:.4f} | {r2_ridge:.4f} | {'N/A':<18}       |") # Accuracy direccional no calculado para Ridge aún
print("---------------------------------------------------")

### 23. Resumen Comparativo Final de Métricas de Modelos

Con la inclusión del modelo Random Forest optimizado, actualizamos la tabla comparativa para tener una visión completa del rendimiento de cada enfoque.

In [ ]:
print("=== Resumen Comparativo de Métricas del Modelo ===")
print("-------------------------------------------------------------------")
print("| Modelo                      | MAE      | RMSE     | R2       | Accuracy Direccional |")
print("-------------------------------------------------------------------")
print(f"| XGBoost                     | {mae_clipped:.4f} | {rmse_clipped:.4f} | {r2_xgb:.4f} | {accuracy_direccional_clipped:.1f}%              |")
print(f"| Random Forest (Inicial)     | {mae_rf:.4f} | {rmse_rf:.4f} | {r2_rf:.4f} | {accuracy_direccional_rf:.1f}%              |")
print(f"| Random Forest (Optimizado)  | {mae_rf_tuned:.4f} | {rmse_rf_tuned:.4f} | {r2_rf_tuned:.4f} | {accuracy_direccional_rf_tuned:.1f}%              |")
print(f"| Ridge Regression            | {mae_ridge:.4f} | {rmse_ridge:.4f} | {r2_ridge:.4f} | {accuracy_direccional_ridge:.1f}%              |")
print("-------------------------------------------------------------------")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# --- 1. Reconstruir Precios Reales y Predichos (RF Optimizado) ---

# Obtener la serie de precios reales del novillo gordo (target original)
precio_real_original_serie = df_ganado_ml['precionovillogordo']

# Obtener el último precio real del set de entrenamiento para iniciar la reconstrucción
last_train_price = precio_real_original_serie.iloc[split_index - 1]

# Los precios reales para el periodo de prueba son simplemente los de la serie original
# desde el split_index en adelante.
precio_real_reconstruido = precio_real_original_serie.iloc[split_index:].reset_index(drop=True)

# Reconstruir la serie de precios predichos encadenando las variaciones del modelo optimizado
precio_predicho_rf_tuned_reconstruido = []
current_base_price = last_train_price # La primera predicción usa el último precio real de entrenamiento

for i in range(len(y_pred_rf_tuned_clipped)):
    predicted_change = y_pred_rf_tuned_clipped[i]
    # Calcular el precio predicho para el mes actual
    current_predicted_price = current_base_price * (1 + predicted_change)
    precio_predicho_rf_tuned_reconstruido.append(current_predicted_price)

    # Para la siguiente iteración, la base es el PRECIO REAL del mes actual (no el predicho),
    # esto evita la propagación acumulativa de errores de predicción en la base.
    current_base_price = precio_real_reconstruido.iloc[i]

# Convertir a Series para que coincida con el índice de fechas
precio_predicho_rf_tuned_reconstruido = pd.Series(precio_predicho_rf_tuned_reconstruido, index=fechas_test)

# --- 2. Generar el Gráfico ---
plt.style.use('seaborn-v0_8-whitegrid') # Aplicar un estilo limpio y profesional

plt.figure(figsize=(16, 8))

# Línea para el Precio Real
plt.plot(fechas_test, precio_real_reconstruido, label='Precio Real', color='#1f77b4', linewidth=2, marker='o', markersize=4)

# Línea para el Precio Predicho (Random Forest Optimizado)
plt.plot(fechas_test, precio_predicho_rf_tuned_reconstruido, label='Precio Predicho (Random Forest Optimizado)', color='#e377c2', linestyle='--', linewidth=2, marker='^', markersize=5)

plt.title('Precio Real vs. Precio Predicho (Random Forest Optimizado) - Novillo Gordo', fontsize=18, pad=20)
plt.xlabel('Fecha', fontsize=14)
plt.ylabel('Precio del Novillo Gordo ($)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper left', fontsize=12)
plt.xticks(rotation=45, ha='right') # Rotar las etiquetas de fecha para mejor legibilidad
plt.tight_layout() # Ajustar el diseño para evitar superposiciones
plt.show()

In [ ]:
# The calculation of accuracy_direccional_ridge was moved to cell 4f121d41
# It was replaced with the summary table which previously included 'N/A'

print("=== Resumen Comparativo de Métricas del Modelo ===")
print("---------------------------------------------------")
print("| Modelo           | MAE      | RMSE     | R2       | Accuracy Direccional |")
print("---------------------------------------------------")
print(f"| XGBoost          | {mae_clipped:.4f} | {rmse_clipped:.4f} | {r2_xgb:.4f} | {accuracy_direccional_clipped:.1f}%              |")
print(f"| Random Forest    | {mae_rf:.4f} | {rmse_rf:.4f} | {r2_rf:.4f} | {accuracy_direccional_rf:.1f}%              |")
print(f"| Ridge Regression | {mae_ridge:.4f} | {rmse_ridge:.4f} | {r2_ridge:.4f} | {accuracy_direccional_ridge:.1f}%              |")
print("---------------------------------------------------")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# --- 1. Variables Necesarias (Asegúrate de que estas variables estén definidas) ---
# 'fechas_test': Fechas del conjunto de prueba. Ya está disponible del split cronológico.
# 'y_test': Variación porcentual REAL del precio en el conjunto de prueba. Ya está disponible.
# 'y_pred_rf': Variación porcentual PREDECIDA por Random Forest. Ya está disponible.
# 'df_ganado_ml': DataFrame con los datos procesados, incluyendo el precio real ('precionovillogordo').
# 'split_index': Índice donde se dividió el set de entrenamiento y prueba.

# --- 2. Reconstruir Precios Reales y Predichos ---

# Obtener la serie de precios reales del novillo gordo (target original)
precio_real_original_serie = df_ganado_ml['precionovillogordo']

# Obtener el último precio real del set de entrenamiento para iniciar la reconstrucción
last_train_price = precio_real_original_serie.iloc[split_index - 1]

# Los precios reales para el periodo de prueba son simplemente los de la serie original
# desde el split_index en adelante.
precio_real_reconstruido = precio_real_original_serie.iloc[split_index:].reset_index(drop=True)

# Reconstruir la serie de precios predichos encadenando las variaciones
precio_predicho_reconstruido = []
current_base_price = last_train_price # La primera predicción usa el último precio real de entrenamiento

for i in range(len(y_pred_rf)):
    predicted_change = y_pred_rf[i]
    # Calcular el precio predicho para el mes actual
    current_predicted_price = current_base_price * (1 + predicted_change)
    precio_predicho_reconstruido.append(current_predicted_price)

    # Para la siguiente iteración, la base es el PRECIO REAL del mes actual (no el predicho),
    # esto evita la propagación acumulativa de errores de predicción en la base.
    current_base_price = precio_real_reconstruido.iloc[i]

# Convertir a Series para que coincida con el índice de fechas
precio_predicho_reconstruido = pd.Series(precio_predicho_reconstruido, index=fechas_test)

# --- 3. Generar el Gráfico ---
plt.style.use('seaborn-v0_8-whitegrid') # Aplicar un estilo limpio y profesional

plt.figure(figsize=(16, 8))

# Línea para el Precio Real
plt.plot(fechas_test, precio_real_reconstruido, label='Precio Real', color='#1f77b4', linewidth=2, marker='o', markersize=4)

# Línea para el Precio Predicho (Random Forest)
plt.plot(fechas_test, precio_predicho_reconstruido, label='Precio Predicho (Random Forest)', color='#9467bd', linestyle='--', linewidth=2, marker='s', markersize=5)

plt.title('Precio Real vs. Precio Predicho (Random Forest) - Novillo Gordo', fontsize=18, pad=20)
plt.xlabel('Fecha', fontsize=14)
plt.ylabel('Precio del Novillo Gordo ($)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper left', fontsize=12)
plt.xticks(rotation=45, ha='right') # Rotar las etiquetas de fecha para mejor legibilidad
plt.tight_layout() # Ajustar el diseño para evitar superposiciones
plt.show()


### 24. Análisis de Estacionalidad y Valores Atípicos (Outliers) en Enero
Vamos a revisar la historia de los datos de entrenamiento para ver si hubo un salto irreal o extremo en algún mes de enero que esté confundiendo a los modelos predictivos.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# 1. Crear un DataFrame temporal para analizar los meses en el conjunto de entrenamiento
df_estacionalidad = pd.DataFrame({
    'Fecha': fechas_train,
    'Variacion_Real': y_train,
    'Mes': fechas_train.dt.month
})

# 2. Boxplot mensual para detectar valores atípicos (outliers)
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_estacionalidad, x='Mes', y='Variacion_Real', hue='Mes', palette='Set3', legend=False)

# Añadir puntos individuales (stripplot) para ver exactamente dónde caen las observaciones
sns.stripplot(data=df_estacionalidad, x='Mes', y='Variacion_Real', color='black', alpha=0.5, size=4)

plt.axhline(0, color='red', linestyle='--', alpha=0.7, label='Línea Cero (Sin Variación)')
plt.title('Distribución Histórica de la Variación del Precio por Mes (Datos de Entrenamiento)', fontsize=15)
plt.xlabel('Mes del Año (1 = Enero)', fontsize=12)
plt.ylabel('Variación Porcentual Real', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

# 3. Filtrar y mostrar los datos extremos de Enero
enero_historico = df_estacionalidad[df_estacionalidad['Mes'] == 1].sort_values(by='Variacion_Real', ascending=False)

print("=== Registro Histórico de Variaciones en ENERO (Entrenamiento) ===")
print("Los valores más altos en la parte superior podrían ser los atípicos que el modelo está intentando replicar:\n")
display(enero_historico.head(10))


### 25. Corrección de Outliers (Capping en Entrenamiento)
Dado que identificamos que los meses de enero de 2010 y 2017 tuvieron alzas anómalas (outliers positivos), vamos a limitar (cap) la variación máxima de enero en nuestro conjunto de entrenamiento a **0%**. Esto evitará que el modelo aprenda a predecir saltos irreales a principios de año, forzándolo a respetar la tendencia natural a la baja.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt

# 1. Copiar y_train para no modificar el original permanentemente
y_train_corregido = y_train.copy()

# 2. Identificar los meses de enero en el set de entrenamiento
es_enero_train = fechas_train.dt.month == 1

# 3. Aplicar Clipping: Limitar el valor máximo de Enero a 0 (cero variación)
# Ya que históricamente (salvo los 2 outliers), enero siempre presenta caída de precios.
limite_superior_enero = 0.0
y_train_corregido[es_enero_train] = np.clip(y_train_corregido[es_enero_train], a_min=None, a_max=limite_superior_enero)

print(f"Se aplicó una corrección: Las variaciones de Enero en entrenamiento mayores a {limite_superior_enero} fueron acotadas a {limite_superior_enero}.\n")

# 4. Reentrenar el Random Forest Optimizado con los datos corregidos
# Usamos los mejores hiperparámetros encontrados en el GridSearch anterior
rf_corregido = RandomForestRegressor(
    n_estimators=150,
    max_depth=5,
    min_samples_leaf=4,
    max_features=0.7,
    random_state=42
)

rf_corregido.fit(X_train, y_train_corregido)

# 5. Predecir sobre el conjunto de prueba (Test)
y_pred_corregido = rf_corregido.predict(X_test)

# 6. Evaluar el nuevo rendimiento
mae_corr = mean_absolute_error(y_test, y_pred_corregido)
rmse_corr = np.sqrt(mean_squared_error(y_test, y_pred_corregido))
acc_dir_corr = np.mean(np.sign(y_test) == np.sign(y_pred_corregido)) * 100
r2_corr = r2_score(y_test, y_pred_corregido)

print("=== Evaluación del RF Optimizado (Entrenamiento Corregido) ===")
print(f"MAE: {mae_corr:.4f}")
print(f"RMSE: {rmse_corr:.4f}")
print(f"R2 Score: {r2_corr:.4f}")
print(f"Accuracy Direccional: {acc_dir_corr:.1f}%")

# 7. Graficar los resultados comparativos
plt.figure(figsize=(14, 6))
plt.plot(fechas_test, y_test, label='Variación Real', color='#1f77b4', linewidth=2, marker='o', markersize=4)
plt.plot(fechas_test, y_pred_corregido, label='Variación Predicha (RF Entrenamiento Corregido)', color='#d62728', linestyle='--', linewidth=2, marker='*', markersize=5)
plt.axhline(0, color='red', linestyle=':', linewidth=2, alpha=0.7, label='Línea Cero')

plt.title('Variación Porcentual: Real vs Predicción (RF con Outliers Corregidos)', fontsize=15, pad=15)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Variación Porcentual Mensual', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()


### 26. Estrategia 2: Añadir Variable 'Mes' y Eliminar Outlier Extremo
Agregamos explícitamente el mes como variable para que el modelo entienda la estacionalidad, y eliminamos las anomalías históricas más grandes (>50% de variación) del set de entrenamiento para limpiar el ruido.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

# 1. Agregar 'Mes' como variable predictora a nuestro conjunto de datos original
X_con_mes = X.copy()
X_con_mes['Mes'] = fechas.dt.month

# 2. Volver a hacer el split cronológico con la nueva variable
X_train_mes = X_con_mes.iloc[:split_index]
X_test_mes = X_con_mes.iloc[split_index:]

# 3. Limpieza profunda: Eliminar el outlier MASIVO histórico
# (En el EDA vimos que en Feb 2010 la variación fue de >70%)
mascara_outliers = y_train < 0.50 # Filtramos valores anómalos mayores al 50%
X_train_limpio = X_train_mes[mascara_outliers]
y_train_limpio = y_train[mascara_outliers]

print(f"Se eliminaron {len(y_train) - len(y_train_limpio)} registros anómalos extremos del entrenamiento.\n")

# 4. Entrenar el Random Forest Optimizado con la nueva variable y datos limpios
rf_mes = RandomForestRegressor(
    n_estimators=150,
    max_depth=5,
    min_samples_leaf=4,
    max_features=0.7,
    random_state=42
)

rf_mes.fit(X_train_limpio, y_train_limpio)

# 5. Predecir sobre Test
y_pred_mes = rf_mes.predict(X_test_mes)

# 6. Evaluar métricas
mae_mes = mean_absolute_error(y_test, y_pred_mes)
rmse_mes = np.sqrt(mean_squared_error(y_test, y_pred_mes))
acc_dir_mes = np.mean(np.sign(y_test) == np.sign(y_pred_mes)) * 100
r2_mes = r2_score(y_test, y_pred_mes)

print("=== Evaluación de la Estrategia 2 (Añadir 'Mes' + Datos Limpios) ===")
print(f"MAE: {mae_mes:.4f}")
print(f"RMSE: {rmse_mes:.4f}")
print(f"R2 Score: {r2_mes:.4f}")
print(f"Accuracy Direccional: {acc_dir_mes:.1f}%")

# 7. Graficar
plt.figure(figsize=(14, 6))
plt.plot(fechas_test, y_test, label='Variación Real', color='#1f77b4', linewidth=2, marker='o', markersize=4)
plt.plot(fechas_test, y_pred_mes, label='Predicción (RF con Mes y sin Outlier)', color='#ff7f0e', linestyle='--', linewidth=2, marker='*', markersize=6)
plt.axhline(0, color='red', linestyle=':', linewidth=2, alpha=0.7, label='Línea Cero')

plt.title('Variación Porcentual: Real vs Predicción (Estrategia 2: Estacionalidad)', fontsize=15, pad=15)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Variación Porcentual Mensual', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

# 8. Ver qué tanta importancia le dio el modelo a la variable 'Mes'
importancias = pd.DataFrame({'Variable': X_train_limpio.columns, 'Importancia': rf_mes.feature_importances_})
importancias = importancias.sort_values('Importancia', ascending=False)
print("\n=== Top 4 Variables Más Importantes ===")
display(importancias.head(4))


### 27. Evaluación de XGBoost con Estrategia 2 (Estacionalidad + Limpieza)
Aplicamos el dataset mejorado al algoritmo XGBoost para comparar su rendimiento frente al Random Forest.

import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 1. Instanciar y Entrenar el Modelo XGBoost con hiperparámetros moderados
xgb_mes = xgb.XGBRegressor(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=5,
    reg_lambda=1,       # Regularización L2 para evitar sobreajuste
    random_state=42
)

# Entrenamos usando los datos limpios de la Estrategia 2
xgb_mes.fit(X_train_limpio, y_train_limpio)

# 2. Generar Predicciones sobre el conjunto de prueba con 'Mes'
y_pred_xgb_mes = xgb_mes.predict(X_test_mes)

# 3. Evaluar métricas
mae_xgb_mes = mean_absolute_error(y_test, y_pred_xgb_mes)
rmse_xgb_mes = np.sqrt(mean_squared_error(y_test, y_pred_xgb_mes))
acc_dir_xgb_mes = np.mean(np.sign(y_test) == np.sign(y_pred_xgb_mes)) * 100
r2_xgb_mes = r2_score(y_test, y_pred_xgb_mes)

print("=== Evaluación de XGBoost (Estrategia 2: Estacionalidad + Datos Limpios) ===")
print(f"MAE: {mae_xgb_mes:.4f}")
print(f"RMSE: {rmse_xgb_mes:.4f}")
print(f"R2 Score: {r2_xgb_mes:.4f}")
print(f"Accuracy Direccional: {acc_dir_xgb_mes:.1f}%")

# 4. Graficar Resultados
plt.figure(figsize=(14, 6))
plt.plot(fechas_test, y_test, label='Variación Real', color='#1f77b4', linewidth=2, marker='o', markersize=4)
plt.plot(fechas_test, y_pred_xgb_mes, label='Predicción (XGBoost con Mes)', color='#2ca02c', linestyle='--', linewidth=2, marker='X', markersize=6)
plt.axhline(0, color='red', linestyle=':', linewidth=2, alpha=0.7, label='Línea Cero')

plt.title('Variación Porcentual: Real vs Predicción (XGBoost - Estrategia 2)', fontsize=15, pad=15)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Variación Porcentual Mensual', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

# 5. Importancia de Variables para XGBoost
importancias_xgb = pd.DataFrame({'Variable': X_train_limpio.columns, 'Importancia': xgb_mes.feature_importances_})
importancias_xgb = importancias_xgb.sort_values('Importancia', ascending=False)
print("\n=== Top 4 Variables Más Importantes (XGBoost) ===")
display(importancias_xgb.head(4))


### 28. Optimización de Hiperparámetros (GridSearchCV) para Estrategia 2
Vamos a aplicar `GridSearchCV` específicamente sobre nuestro conjunto de datos limpio que incluye la variable estacional `Mes` (`X_train_limpio`, `y_train_limpio`) para encontrar la mejor versión posible del Random Forest.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt

print("--- Iniciando Búsqueda de Hiperparámetros con GridSearchCV para RF (Estrategia 2) ---")

# 1. Definir la rejilla de parámetros a explorar
param_grid_estrategia2 = {
    'n_estimators': [100, 150, 200],
    'max_depth': [4, 5, 6, 8],
    'min_samples_leaf': [2, 4, 6],
    'max_features': [0.7, 0.8, 1.0]
}

# 2. Instanciar el modelo base
rf_base_est2 = RandomForestRegressor(random_state=42)

# 3. Configurar GridSearchCV
grid_search_est2 = GridSearchCV(
    estimator=rf_base_est2,
    param_grid=param_grid_estrategia2,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)

# 4. Ejecutar la búsqueda con los datos limpios y la variable 'Mes'
grid_search_est2.fit(X_train_limpio, y_train_limpio)

# 5. Extraer los mejores parámetros
best_params_est2 = grid_search_est2.best_params_
print(f"\n--- Mejores Hiperparámetros Encontrados: {best_params_est2} ---")

# 6. Obtener el modelo ya optimizado
rf_tuned_est2 = grid_search_est2.best_estimator_

# 7. Generar predicciones sobre el conjunto de prueba (que también incluye 'Mes')
y_pred_tuned_est2 = rf_tuned_est2.predict(X_test_mes)

# 8. Evaluar métricas finales
mae_tuned_est2 = mean_absolute_error(y_test, y_pred_tuned_est2)
rmse_tuned_est2 = np.sqrt(mean_squared_error(y_test, y_pred_tuned_est2))
r2_tuned_est2 = r2_score(y_test, y_pred_tuned_est2)
acc_dir_tuned_est2 = np.mean(np.sign(y_test) == np.sign(y_pred_tuned_est2)) * 100

print("\n=== Evaluación del Modelo Random Forest Optimizado (Estrategia 2) ===")
print(f"MAE: {mae_tuned_est2:.4f}")
print(f"RMSE: {rmse_tuned_est2:.4f}")
print(f"R2 Score: {r2_tuned_est2:.4f}")
print(f"Accuracy Direccional: {acc_dir_tuned_est2:.1f}%")

# 9. Gráfico de Predicción vs Realidad
plt.figure(figsize=(14, 6))
plt.plot(fechas_test, y_test, label='Variación Real', color='#1f77b4', linewidth=2, marker='o', markersize=4)
plt.plot(fechas_test, y_pred_tuned_est2, label='Predicción (RF Optimizado Est. 2)', color='#e377c2', linestyle='--', linewidth=2, marker='^', markersize=6)
plt.axhline(0, color='red', linestyle=':', linewidth=2, alpha=0.7, label='Línea Cero')

plt.title('Variación Porcentual: Real vs Predicción (Random Forest Optimizado - Estrategia 2)', fontsize=15, pad=15)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Variación Porcentual Mensual', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()


### 29. Exportación del Modelo para Despliegue Web
Para usar el modelo en producción (ej. una API con FastAPI o Flask), necesitamos guardarlo en un archivo. Guardaremos tanto el modelo optimizado como la lista de características (variables) que este espera recibir.

In [ ]:
import joblib
import os

# 1. Definir los nombres de los archivos
nombre_archivo_modelo = 'rf_modelo_produccion.joblib'
nombre_archivo_vars = 'rf_variables_produccion.joblib'

# 2. Rutas completas en Google Drive
ruta_modelo = os.path.join(PATH, nombre_archivo_modelo)
ruta_vars = os.path.join(PATH, nombre_archivo_vars)

# 3. Guardar el modelo optimizado de la Estrategia 2
joblib.dump(rf_tuned_est2, ruta_modelo)

# 4. Guardar la lista de variables exactas que el modelo requiere
variables_modelo = list(X_train_limpio.columns)
joblib.dump(variables_modelo, ruta_vars)

print(f"=== Exportación Exitosa ===")
print(f"Modelo guardado en: {ruta_modelo}")
print(f"Variables guardadas en: {ruta_vars}")
print("\nVariables que el modelo web deberá solicitar:")
for i, var in enumerate(variables_modelo):
    print(f"{i+1}. {var}")

In [ ]:
%%writefile app.py
import streamlit as st
import joblib
import pandas as pd

# 1. Configuración de la página
st.set_page_config(page_title="Demo: Predictor Novillo Gordo", layout="centered")
st.title("🐄 Predictor de Precios: Novillo Gordo")
st.markdown("### Prototipo para Presentación")
st.info("Ingresa los datos históricos (rezagos) en el panel izquierdo para simular la variación del precio.")

# 2. Cargar el modelo
@st.cache_resource
def cargar_modelo():
    # Usamos la ruta exacta donde guardamos el modelo optimizado
    return joblib.load('/content/drive/MyDrive/Colab Notebooks/archivos_modelo/rf_modelo_produccion.joblib')

try:
    modelo = cargar_modelo()
except Exception as e:
    st.error(f"Error al cargar el modelo. Verifica que tu Drive esté montado. Detalles: {e}")
    st.stop()

# 3. Panel lateral para ingreso de variables
st.sidebar.header("Variables del Mercado")

# Variable Estacional
mes = st.sidebar.slider("Mes a predecir", 1, 12, 10)

# Variables de Oferta (Lags biológicos)
st.sidebar.subheader("Oferta (Cantidad Histórica)")
cant_terneros_lag8 = st.sidebar.number_input("Terneros (Hace 8 meses)", value=6000)
cant_novillos_engorda_lag4 = st.sidebar.number_input("Novillos Engorda (Hace 4 meses)", value=1500)
cant_terneros_lag12 = st.sidebar.number_input("Terneros (Hace 12 meses)", value=5500)

# Variables Económicas (Lags del Dólar)
st.sidebar.subheader("Economía (Dólar Real)")
dolar_real_lag1 = st.sidebar.number_input("Dólar (Hace 1 mes)", value=900.0)
dolar_real_lag2 = st.sidebar.number_input("Dólar (Hace 2 meses)", value=890.0)
dolar_real_lag3 = st.sidebar.number_input("Dólar (Hace 3 meses)", value=880.0)

# Variables Climáticas (Lags de Precipitaciones)
st.sidebar.subheader("Clima (Precipitaciones mm)")
precip_lag1 = st.sidebar.number_input("Precipitaciones (Hace 1 mes)", value=100.0)
precip_lag2 = st.sidebar.number_input("Precipitaciones (Hace 2 meses)", value=120.0)

# 4. Botón de cálculo y despliegue de resultados
if st.button("Calcular Predicción", type="primary"):
    # Crear DataFrame con los nombres exactos requeridos por el modelo
    datos_entrada = pd.DataFrame({
        'cant_terneros_lag8': [cant_terneros_lag8],
        'cant_novillos_engorda_lag4': [cant_novillos_engorda_lag4],
        'cant_terneros_lag12': [cant_terneros_lag12],
        'dolar_real_lag1': [dolar_real_lag1],
        'dolar_real_lag2': [dolar_real_lag2],
        'dolar_real_lag3': [dolar_real_lag3],
        'precipitaciones_lag1': [precip_lag1],
        'precipitaciones_lag2': [precip_lag2],
        'Mes': [mes]
    })

    # Realizar predicción de la variación porcentual
    variacion = modelo.predict(datos_entrada)[0]

    st.markdown("---")
    st.subheader("Resultado de la Predicción")

    # Para propósitos de la demo, simulamos un precio base
    precio_base_demo = 2000
    precio_estimado = precio_base_demo * (1 + variacion)

    col1, col2 = st.columns(2)
    col1.metric("Variación Porcentual Esperada", f"{variacion * 100:.2f}%")
    col2.metric(f"Precio Estimado (Base Ref: ${precio_base_demo})", f"${precio_estimado:,.0f} CLP")

    st.success("¡Análisis ejecutado correctamente en base al modelo Random Forest!")


In [ ]:
import os
import time
import subprocess
import sys
from google.colab import output

# 1. Matar procesos anteriores en el puerto 8501
print("Limpiando procesos anteriores...")
os.system("fuser -k 8501/tcp")
time.sleep(2) # Esperar a que se libere el puerto

# 2. Iniciar Streamlit en segundo plano
print("Iniciando Streamlit en el puerto 8501...")
subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(5) # Esperar unos segundos para asegurar que la app levante

# 3. Generar el iframe seguro usando el proxy interno de Colab
print("Cargando la interfaz de Streamlit... (Puede tardar unos segundos)")
output.serve_kernel_port_as_iframe(8501, height=800)


In [ ]:
from google.colab import output

print("Haz clic en el siguiente enlace para abrir la aplicación en una nueva pestaña:")
# Esta función genera un enlace directo al puerto 8501 de tu entorno virtual en Colab
print(output.eval_js("google.colab.kernel.proxyPort(8501)"))

### Método Alternativo: Localtunnel
Si el enlace directo de Colab no carga la aplicación correctamente, podemos usar Localtunnel.

In [ ]:
# 1. Instalamos localtunnel
!npm install -g localtunnel

In [ ]:
pip install streamlit

In [ ]:
# 1. Instalar pyngrok si es necesario
!pip install pyngrok

import subprocess
import sys
import time
from pyngrok import ngrok

# 2. Configurar tu token de Ngrok
# Reemplaza "TU_AUTHTOKEN_AQUI" con tu token real de ngrok
ngrok.set_auth_token("3F94PzYeWatsyCso81ECLHAUNwR_535UkAUfdgkdb6FTP1JVk")

# 3. Iniciar Streamlit en segundo plano en el puerto 8501
print("Iniciando Streamlit en el puerto 8501...")
subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(5) # Esperar unos segundos para asegurar que la app levante

# 4. Abrir el túnel ngrok al puerto 8501
print("Abriendo el túnel ngrok...")
public_url = ngrok.connect(8501).public_url
print("Tu aplicación Streamlit está disponible en la URL pública:", public_url)

# Nota: El comando `!npx localtunnel` ha sido reemplazado por la configuración de pyngrok.

In [ ]:
import urllib
print("COPIA ESTA CONTRASEÑA/IP:")
print(urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))

In [ ]:
# 3. Ejecutamos Streamlit y Localtunnel al mismo tiempo
# Haz clic en el enlace que dice "your url is: https://..." y pega la contraseña de arriba.
from pyngrok import ngrok

# Configura tu token (solo se hace una vez)
ngrok.set_auth_token("TU_AUTHTOKEN_AQUI")

# Abre el túnel en el puerto que usa tu app (ej. 8501 para Streamlit)
public_url = ngrok.connect(8501)
print("Tu app está corriendo en:", public_url)

In [ ]:
import joblib
import pandas as pd
import traceback

print("--- Test de Predicción (Replicando app.py) ---")
try:
    # 1. Cargar el modelo
    modelo_test = joblib.load('/content/drive/MyDrive/Colab Notebooks/archivos_modelo/rf_modelo_produccion.joblib')
    print("✅ Modelo cargado correctamente.")

    # 2. Crear datos de prueba idénticos a los de app.py
    datos_entrada_test = pd.DataFrame({
        'cant_terneros_lag8': [6000],
        'cant_novillos_engorda_lag4': [1500],
        'cant_terneros_lag12': [5500],
        'dolar_real_lag1': [900.0],
        'dolar_real_lag2': [890.0],
        'dolar_real_lag3': [880.0],
        'precipitaciones_lag1': [100.0],
        'precipitaciones_lag2': [120.0],
        'Mes': [10]
    })
    print("✅ Datos de entrada creados:")
    display(datos_entrada_test)

    # 3. Intentar predecir
    variacion_test = modelo_test.predict(datos_entrada_test)[0]
    print(f"\n✅ Predicción exitosa: {variacion_test}")

except Exception as e:
    print("\n❌ ERROR ENCONTRADO AL PREDECIR:")
    traceback.print_exc()